In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import*

In [0]:
df=spark.table('workspace.bronze.dwh_cust_az12')
df.display()

In [0]:
df.filter(col('BDate')>=current_date()).display()


**## Trim String Fields**

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType,StringType):
        df=df.withColumn(field.name,trim(col(field.name)))
df.display()



## **Standardize Gender**

In [0]:
df=df.withColumn('GEN',when(upper(col('GEN')).isin('M','MALE'),'Male')
              .when(upper(col('GEN')).isin('F','FEMALE'),'Female').
              otherwise('N/A'))


In [0]:

df.groupBy('GEN').count().display()

## **Clean Bdate**

In [0]:
df=df.withColumn('BDATE',when(col('BDATE')>current_date(),None).otherwise(col("BDATE")))
df.filter(col('BDATE')>=current_date()).display()


## **Standardize  CID**

In [0]:
df.filter(~col('CID').startswith('NAS')).display()

In [0]:
df=df.withColumn('CID',when(col('CID').startswith('NAS'),substring(col('CID'),4,length(col('CID')))).otherwise(col("CID")))

In [0]:
df.filter(col('CID').startswith('NAS')).display()

**Renaming Columns**

In [0]:
rename={'CID':'customer_number','BDATE':'birthdate','GEN':'gender'}
df=df.select([col(column).alias(rename.get(column,column)) for column in df.columns])
df.display()


In [0]:
df.write.mode('overwrite').format('delta').saveAsTable('workspace.silver.erp_customers1')